<a href="https://colab.research.google.com/github/mayait/CursoAnalisisDatos_IA_2026/blob/main/sitio/labs/lab_03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Laboratorio 3 · Estadística descriptiva sin mentir

La semana pasada decidiste la unidad de análisis y viste que «cuántos clientes activos tenemos» tiene
siete respuestas. Hoy la unidad ya está fijada —la factura— y el problema es otro: qué número la
resume. El promedio es la estadística más usada del mundo y la que más decisiones arruina. En
Comercial Andina el ticket promedio es 180,84 y el ticket mediano 30,08; hoy averiguas cuál de los
dos le corresponde a un cliente real y por qué la diferencia no es un error de cálculo sino un
hallazgo de negocio. Esto desbloquea la línea base descriptiva contra la que se compara todo lo que
viene después.

> **Hoy haces** · Perfilas la base completa con `describe`, `quantile` y `value_counts` (90 min).
> Demuestras con un histograma que el ticket tiene dos jorobas, separas las dos poblaciones que las
> producen y cuantificas la dispersión de cada una. Cierras con doce preguntas descriptivas contra
> reloj.
>
> **Entrega** · Este cuaderno ejecutado, los tres ejercicios resueltos y las doce preguntas
> respondidas, cada una indicando si el estadístico correcto es la media o la mediana y por qué.
> Nombre de archivo: `lab_03_apellido.ipynb`.

In [ ]:
# --- Setup del entorno ---
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 4)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

# Los datos de Comercial Andina viven en sitio/datos/
BASE_URL = ("https://raw.githubusercontent.com/mayait/"
            "CursoAnalisisDatos_IA_2026/main/sitio/datos")
ARCHIVOS = ["clientes.csv", "productos.csv", "sucursales.csv", "ventas.csv",
            "ventas_limpias.csv", "marketing_mensual.csv",
            "experimento_reactivacion.csv"]
CANDIDATOS = [Path("../datos"), Path("datos"), Path("sitio/datos"),
              Path("/content/CursoAnalisisDatos_IA_2026/sitio/datos")]
DATOS = next((p for p in CANDIDATOS if (p / "ventas.csv").exists()), None)
if DATOS is None:
    # En Colab el cuaderno llega solo: se descargan los siete archivos una vez.
    from urllib.request import urlretrieve
    DATOS = Path("datos")
    DATOS.mkdir(exist_ok=True)
    for archivo in ARCHIVOS:
        if not (DATOS / archivo).exists():
            urlretrieve(f"{BASE_URL}/{archivo}", DATOS / archivo)

print("Setup completo ✓")
print(f"pandas {pd.__version__} · datos en {DATOS.resolve()}")

## 1. Preparación mínima

Antes de describir hay que decidir qué entra en el cálculo. Hoy se hacen tres cosas y ninguna más:
quitar los duplicados exactos, convertir la fecha y separar las devoluciones de las ventas. No es una
limpieza —eso es la semana 4— es lo mínimo para que los estadísticos signifiquen algo. Cada decisión
queda escrita en el código, que es donde se auditan las decisiones.

In [ ]:
ventas = pd.read_csv(DATOS / "ventas.csv")
clientes = pd.read_csv(DATOS / "clientes.csv")
productos = pd.read_csv(DATOS / "productos.csv")

antes = len(ventas)
ventas = ventas.drop_duplicates()                       # 1) duplicados exactos fuera
ventas["fecha"] = pd.to_datetime(ventas["fecha"], format="mixed", dayfirst=True)   # 2) fecha real
ventas["monto"] = ventas["cantidad"] * ventas["precio_unitario"] * (1 - ventas["descuento"])

lineas = ventas[~ventas["es_devolucion"]]               # 3) las devoluciones van aparte
ticket = lineas.groupby("factura_id")["monto"].sum()    # unidad de análisis: la factura

print(f"filas leídas          : {antes:,}")
print(f"filas tras duplicados : {len(ventas):,}  (se quitaron {antes - len(ventas):,})")
print(f"líneas de venta       : {len(lineas):,}")
print(f"facturas de venta     : {len(ticket):,}")

## 2. `describe`: el perfilado que cabe en una línea

`describe()` sobre el DataFrame entero es el primer gesto de cualquier perfilado. Con
`include="all"` incluye también las columnas de texto, y ahí aparecen `count`, `unique`, `top` y
`freq`, que son las que sirven para categorías.

In [ ]:
ventas.describe()

In [ ]:
ventas.describe(include="all").T

Léelo por filas, no por columnas. `cantidad` va de -59 a 59 con media 15,94 y mediana 4: hay negativos, y la
media está a cuatro veces la mediana. `precio_unitario` llega a 59,10 cuando el producto más caro del
catálogo cuesta 5,91: alguien tecleó un cero de más. Y `count` no es igual en todas las columnas:
`cliente_id` y `precio_unitario` tienen huecos. Todo eso es material de la semana 4; hoy solo lo
anotas.

## 3. La media que no le corresponde a nadie

Ahora el estadístico que va a llegar al comité: el ticket promedio.

In [ ]:
print(ticket.describe().to_string())
print()
print(f"media   : {ticket.mean():>10,.2f}")
print(f"mediana : {ticket.median():>10,.2f}")
print(f"razón media/mediana : {ticket.mean() / ticket.median():.2f}")
print(f"facturas por debajo de la media : {(ticket < ticket.mean()).mean():.1%}")

📌 **La media es 6,01 veces la mediana y el 70,0 % de las facturas queda por debajo de la media.** En
una distribución simétrica ese porcentaje sería 50 %. Cuando media y mediana se separan tanto, la
media dejó de describir el centro: describe el efecto de una minoría sobre la suma. Mírala.

In [ ]:
fig, ax = plt.subplots()
ax.hist(ticket, bins=80, color="#4C72B0", edgecolor="white", linewidth=0.4)
ax.axvline(ticket.median(), color="#55A868", linewidth=2.5, label=f"mediana {ticket.median():,.2f}")
ax.axvline(ticket.mean(), color="#C44E52", linewidth=2.5, linestyle="--", label=f"media {ticket.mean():,.2f}")
ax.set_title("El ticket tiene dos jorobas: la media cae en el valle donde casi no hay facturas")
ax.set_xlabel("monto de la factura")
ax.set_ylabel("número de facturas")
ax.legend()
plt.tight_layout()
plt.show()

La media —línea roja— no cae sobre ninguna de las dos jorobas: cae en el hueco que las separa. Ese es
el sentido literal de «el promedio no le corresponde a ningún cliente real». Y no es una cola larga
cualquiera: son **dos poblaciones distintas metidas en el mismo archivo**.

## 4. Las dos poblaciones

La hipótesis obvia es que las jorobas son los dos tipos de cliente. Se comprueba en dos líneas.

In [ ]:
facturas = (lineas.dropna(subset=["cliente_id"])
            .groupby("factura_id")
            .agg(monto=("monto", "sum"), cliente_id=("cliente_id", "first"), fecha=("fecha", "min"))
            .reset_index()
            .merge(clientes[["cliente_id", "tipo_cliente", "ciudad"]], on="cliente_id", how="left"))

print(f"facturas con cliente identificado: {len(facturas):,} de {len(ticket):,} "
      f"({len(facturas) / len(ticket):.1%})")

perfil = facturas.groupby("tipo_cliente")["monto"].agg(
    facturas="count", media="mean", mediana="median", desviacion="std",
    p05=lambda s: s.quantile(0.05), p95=lambda s: s.quantile(0.95))
perfil["CV"] = perfil["desviacion"] / perfil["media"]
perfil

In [ ]:
fig, ax = plt.subplots()
for tipo, color in [("Minorista", "#4C72B0"), ("Mayorista", "#DD8452")]:
    ax.hist(facturas.loc[facturas["tipo_cliente"] == tipo, "monto"],
            bins=60, alpha=0.75, label=tipo, color=color)
ax.axvline(ticket.mean(), color="#C44E52", linewidth=2.5, linestyle="--",
           label=f"media global {ticket.mean():,.2f}")
ax.set_title("Cada joroba es un tipo de cliente: la media global no describe a ninguno de los dos")
ax.set_xlabel("monto de la factura")
ax.set_ylabel("número de facturas")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.2))
sns.boxplot(data=facturas, x="monto", y="tipo_cliente", ax=ax, showfliers=False)
ax.set_title("Las cajas no se tocan: el 95 % de los minoristas está por debajo del 5 % de los mayoristas")
ax.set_xlabel("monto de la factura")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

print(f"p95 de minoristas : {facturas.loc[facturas['tipo_cliente'] == 'Minorista', 'monto'].quantile(0.95):,.2f}")
print(f"p05 de mayoristas : {facturas.loc[facturas['tipo_cliente'] == 'Mayorista', 'monto'].quantile(0.05):,.2f}")

📌 Los mayoristas son el **21,6 %** de los clientes del padrón, hacen el **34,8 %** de las facturas y
aportan el **92,1 %** de la facturación. Dentro de cada grupo la media sí sirve: minoristas 20,32 con
mediana 18,61, mayoristas 442,93 con mediana 413,46. Media y mediana casi coinciden porque cada
población, por separado, es razonablemente simétrica. **El problema nunca fue la media: fue promediar
dos cosas distintas.**

In [ ]:
print(f"mayoristas: {(clientes['tipo_cliente'] == 'Mayorista').mean():.1%} de los clientes del padrón")
print(f"            {(facturas['tipo_cliente'] == 'Mayorista').mean():.1%} de las facturas")
print(f"            {facturas.loc[facturas['tipo_cliente'] == 'Mayorista', 'monto'].sum() / facturas['monto'].sum():.1%} de la facturación")

## 5. Percentiles: `quantile` y su lectura de negocio

Un percentil no es un adorno estadístico: es un umbral operativo. El p75 del ticket contesta «¿desde
qué monto una factura está en el 25 % más grande?», que es exactamente la pregunta que hace falta para
definir un beneficio, una alerta o un criterio de atención preferente.

In [ ]:
cortes = [0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
percentiles = pd.DataFrame({
    "global": ticket.quantile(cortes),
    "minoristas": facturas.loc[facturas["tipo_cliente"] == "Minorista", "monto"].quantile(cortes),
    "mayoristas": facturas.loc[facturas["tipo_cliente"] == "Mayorista", "monto"].quantile(cortes),
})
percentiles.index = [f"p{int(c * 100):02d}" for c in cortes]
percentiles

La columna «global» es un frankenstein: salta de 16,76 en p25 a 290,72 en p75 porque entre esos dos
puntos está el vacío entre las dos poblaciones. Las columnas por tipo, en cambio, se leen sin
sobresaltos. Regla práctica: **si tus percentiles dan un salto, tienes más de una población.**

## 6. Frecuencias: `value_counts`

Para las variables categóricas no hay media que valga. `value_counts()` da el conteo y
`value_counts(normalize=True)` la proporción, que es casi siempre lo que quieres reportar.

In [ ]:
print("Descuento aplicado (proporción de líneas)")
print((lineas["descuento"].value_counts(normalize=True).sort_index() * 100).round(1).to_string())
print()
print("Sucursal (proporción de líneas)")
print((lineas["sucursal_id"].value_counts(normalize=True) * 100).round(1).to_string())
print()
print(f"líneas con algún descuento: {(lineas['descuento'] > 0).mean():.1%}")

Tres cuartas partes de las líneas van sin descuento y la sucursal en línea (`S99`) mueve el 26,0 % de
las líneas, casi tanto como Quito. Ese dato no aparece en ningún promedio.

**La categoría «Otros».** Cuando una variable tiene muchos niveles se agrupa la cola en «Otros». Es
útil y es peligroso: si el hallazgo está en la cola, acabas de esconderlo.

In [ ]:
por_subcat = (lineas.merge(productos, on="producto_id", how="left")
              .groupby("subcategoria")["monto"].sum().sort_values(ascending=False))

top5 = por_subcat.head(5)
resumida = pd.concat([top5, pd.Series({"Otros (7 subcategorías)": por_subcat.iloc[5:].sum()})])

print(f"subcategorías en el catálogo : {por_subcat.size}")
print(f"cuota del top 5              : {top5.sum() / por_subcat.sum():.1%}")
print()
print(resumida.to_string())

## 7. Dispersión: desviación, rango intercuartílico y coeficiente de variación

El centro sin la dispersión es media información. Dos ciudades con el mismo ticket mediano y
dispersión distinta necesitan políticas comerciales distintas. La desviación estándar está en la
unidad del dato —dólares— y por eso no se puede comparar entre grupos de tamaño distinto; el
**coeficiente de variación** (desviación dividida por media) no tiene unidades y sí se puede comparar.

In [ ]:
def dispersion(s):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    return pd.Series({"n": s.size, "media": s.mean(), "mediana": s.median(),
                      "desviación": s.std(), "IQR": q3 - q1, "CV": s.std() / s.mean()})

tabla = pd.DataFrame({
    "todas las facturas": dispersion(ticket),
    "minoristas": dispersion(facturas.loc[facturas["tipo_cliente"] == "Minorista", "monto"]),
    "mayoristas": dispersion(facturas.loc[facturas["tipo_cliente"] == "Mayorista", "monto"]),
})
tabla

📌 **CV global 1,50; CV de minoristas 0,62; CV de mayoristas 0,59.** Un CV por encima de 1 significa
que la desviación supera a la media: el número medio no sirve ni para planificar inventario ni para
fijar una meta. Al separar las poblaciones, el CV cae a la mitad en ambos grupos. La misma cifra que
delata el problema confirma que la solución fue la correcta.

## 8. Tamaño de muestra: por qué un promedio sobre doce observaciones no es un promedio

Si un vendedor reporta el ticket promedio de sus doce facturas del mes, ¿cuánta de esa cifra es
desempeño y cuánta es azar? Se responde sacando muestras de la misma población y viendo cuánto se
mueve el promedio.

In [ ]:
poblacion = facturas.loc[facturas["tipo_cliente"] == "Minorista", "monto"].values
generador = np.random.default_rng(SEED)

filas = []
for n in (12, 50, 200, 1000):
    medias = [generador.choice(poblacion, size=n, replace=False).mean() for _ in range(500)]
    filas.append({"tamaño de muestra": n, "promedio de los promedios": np.mean(medias),
                  "el más bajo": np.min(medias), "el más alto": np.max(medias),
                  "desviación de la media": np.std(medias)})

pd.DataFrame(filas).set_index("tamaño de muestra")

Con doce facturas el promedio muestral se mueve entre 11,40 y 30,86 aunque la población sea siempre la
misma: un vendedor podría reportar el triple que otro sin haber hecho nada distinto. Con mil facturas
el rango se cierra entre 18,55 y 21,21. **Antes de premiar o castigar un promedio, cuenta cuántas
observaciones lo sostienen.**

### 🌶️ Ejercicio 1 — Guiado

Repite la sección 4 completa pero abriendo por `ciudad` en lugar de por `tipo_cliente`: tabla con
media, mediana, desviación y CV por ciudad, y un histograma o un boxplot. Después responde: ¿hay
también dos poblaciones dentro de cada ciudad? ¿Cómo lo sabes?

In [ ]:
# TU CÓDIGO AQUÍ
# Pista 1: la columna ciudad viene con variantes de escritura; normalízala primero con
#          .str.strip().str.title() y arregla "Guayaquíl" a mano.
# Pista 2: reutiliza la función dispersion() de la sección 7 con groupby(...).apply()

### 🔥 Desafío

El `describe()` del ticket dice que el mínimo es **-693,68**. Una factura de venta no puede ser
negativa: las devoluciones ya fueron excluidas. Encuentra esas facturas, cuenta cuántas son, calcula
cuánto pesan sobre la media global y decide, con argumento escrito, si deben entrar o no en la línea
base descriptiva del negocio.

In [ ]:
# TU CÓDIGO AQUÍ
# Pista 1: ticket[ticket < 0]
# Pista 2: el origen está en lineas["cantidad"] < 0 con es_devolucion == False
# Pista 3: compara ticket.mean() con ticket[ticket > 0].mean() y reporta la diferencia

### 🎯 Reto en clase (15 min)

En equipos, contra reloj. Doce preguntas de dificultad creciente sobre la base del curso. Para **cada
respuesta** hay que indicar si el estadístico correcto es la media o la mediana y justificarlo con la
forma de la distribución, no con la costumbre.

1. ¿Cuántas facturas de venta hay y qué periodo cubren?
2. ¿Cuál es el ticket mediano y cuál el promedio? ¿Cuál reportas al comité y por qué?
3. ¿Qué proporción de las líneas lleva algún descuento y cuál es el descuento más frecuente?
4. ¿Cuál es la categoría con más líneas vendidas? ¿Y la de mayor facturación? ¿Coinciden?
5. ¿Cuál es el rango intercuartílico del ticket de los minoristas y qué te dice de la mitad central?
6. ¿Qué sucursal concentra más líneas? ¿Cambia el ranking si mides facturación en lugar de líneas?
7. ¿Cuántas facturas hacen falta, ordenadas de mayor a menor, para acumular el 50 % de la facturación?
8. ¿Cuál es el CV del ticket por ciudad? ¿Qué ciudad tiene la demanda más predecible?
9. ¿En qué percentil cae una factura de 100,00 dentro de los minoristas? ¿Y dentro de los mayoristas?
10. ¿Cuál es el ticket mediano por ciudad? ¿La ciudad con la mediana más alta es la que más factura?
11. Compara la dispersión del ticket medio de los clientes con 3 facturas o menos contra la de los que
    tienen más de 25. ¿Qué implica para un ranking de «mejores clientes»?
12. `value_counts()` sobre `producto_id` da 74 valores. Si agrupas todo salvo el top 10 en «Otros»,
    ¿qué porcentaje de la facturación queda escondido en esa categoría? ¿Es aceptable?

In [ ]:
# TU CÓDIGO AQUÍ
# Pista: una celda por pregunta, con la respuesta impresa y una línea de comentario
# que diga "media" o "mediana" y por qué. Las preguntas 7 y 11 son las que más tiempo toman.

## La trampa de hoy

⚠️ **Reportar la media de una distribución con cola larga.** Marketing quiere lanzar «envío gratis
desde el ticket promedio» y pide la cifra. Le entregas 180,84 sin mirar la forma de la distribución.
Esto es lo que ocurre.

In [ ]:
umbral_media = ticket.mean()
umbral_mediana = ticket.median()

minoristas = facturas.loc[facturas["tipo_cliente"] == "Minorista", "monto"]
mayoristas = facturas.loc[facturas["tipo_cliente"] == "Mayorista", "monto"]

comparacion = pd.DataFrame({
    "umbral": [umbral_media, umbral_mediana],
    "% de facturas que lo alcanzan": [(ticket >= umbral_media).mean() * 100,
                                      (ticket >= umbral_mediana).mean() * 100],
    "% de minoristas que lo alcanzan": [(minoristas >= umbral_media).mean() * 100,
                                        (minoristas >= umbral_mediana).mean() * 100],
    "% de mayoristas que lo alcanzan": [(mayoristas >= umbral_media).mean() * 100,
                                        (mayoristas >= umbral_mediana).mean() * 100],
}, index=["umbral = media (180,84)", "umbral = mediana (30,08)"])

print(comparacion.round(2).to_string())
print()
print(f"minoristas que alcanzan la media: {(minoristas >= umbral_media).sum()} de {minoristas.size:,}")

Dos de diez mil trescientas sesenta y cuatro. La promoción se anuncia como si estuviera al alcance
del cliente típico y en realidad está por encima del percentil 99 del 78 % de la base. El presupuesto
se gasta, la conversión no se mueve y en la reunión de cierre alguien concluye que «al cliente
ecuatoriano no le importa el envío gratis».

Los dos números salieron del mismo archivo, con la misma línea de código, cambiando `mean` por
`median`. La media no está mal calculada: está mal elegida. Y la razón por la que está mal elegida
cabe en un histograma que tardó cuatro segundos en dibujarse.

## Entregable

Sube `lab_03_apellido.ipynb` con:

- El cuaderno ejecutado, con el histograma de la sección 3 y el de la sección 4.
- Los tres ejercicios resueltos, incluido el argumento escrito del desafío sobre las facturas negativas.
- Las doce preguntas respondidas, cada una con su estadístico elegido y una línea de justificación.
- Tres observaciones descriptivas que un gerente podría accionar mañana, con la cifra que las respalda.
- Una fila nueva en la bitácora: pídele al asistente que genere doce preguntas descriptivas sobre tu
  propio dataset y verifica una por una la respuesta que te dio contra lo que devuelve pandas. Al
  menos una no va a coincidir; anota cuál y por qué.

## Para tu equipo

- El perfilado descriptivo del dataset del caso es el entregable de esta semana. Empiecen por
  `describe(include="all")` y busquen el salto de percentiles: si lo encuentran, tienen dos
  poblaciones y el proyecto acaba de mejorar.
- Ninguna cifra del informe del grupo va sola: media o mediana **siempre** acompañada de una medida de
  dispersión y del número de observaciones que la sostienen.
- Si su dataset tiene menos de cien filas por grupo comparado, no reporten promedios por grupo todavía.
  Anótenlo como limitación conocida en la ficha de análisis y sigan.